In [1]:
import json
import os

label_root = '/home/hocheol/inskin_ai/no_track/datasets/age_data/v0.3/v0.3.0/114_filtered/label'
data_dict = {}
for phase in ['train', 'val', 'test']:
    with open(os.path.join(label_root, f"age_{phase}.json"), 'r') as f:
        data_dict[phase] = json.load(f)
len(data_dict['train']), len(data_dict['val']), len(data_dict['test']), sum([len(data_dict[phase]) for phase in ['train', 'val', 'test']])

(47527, 5730, 6166, 59423)

In [2]:
# size는 각각 재고, path는 각 이미지 경로로 새로 만들고 나머지는 그대로 가져오기: user_id, gender, age, age_class
data_dict['train'][0]

{'user_id': '0166',
 'path': '114_filtered/data/0166_Phone_High_001.jpg',
 'gender': 'male',
 'age': '20s',
 'age_class': 1,
 'size': {'height': 833, 'width': 732, 'channels': 3},
 'sector_boxes': {'forehead': {'y_min': 0,
   'x_min': 121,
   'y_max': 172,
   'x_max': 606},
  'right_eye': {'y_min': 217, 'x_min': 41, 'y_max': 409, 'x_max': 294},
  'left_eye': {'y_min': 218, 'x_min': 441, 'y_max': 411, 'x_max': 689},
  'nasolabial': {'y_min': 341, 'x_min': 136, 'y_max': 603, 'x_max': 596},
  'oral': {'y_min': 507, 'x_min': 146, 'y_max': 753, 'x_max': 587}}}

In [3]:
id_sets = {'train': set(), 'val': set(), 'test': set()}

for phase in ['train', 'val', 'test']:
    for item in data_dict[phase]:
        id_sets[phase].add(item['user_id'])

merged_ids = id_sets['train'] | id_sets['val'] | id_sets['test']

len(id_sets['train']), len(id_sets['val']), len(id_sets['test']), len(merged_ids)

(997, 125, 125, 1247)

In [ ]:
import os
from preprocess.face_crop_roll import age_preprocess
import utils.Mediapipe as mp
import matplotlib.pyplot as plt
from PIL import Image
import cv2

from utils.slack import send_slack_message

from tqdm import tqdm

ORIGIN_ROOT = '/home/hocheol/inskin_ai/no_track/datasets/origin_data-simple/114_data'
FILTERED_ROOT = '/home/hocheol/inskin_ai/no_track/datasets/origin_data/114_data/114_filtered'
REMAINED_ROOT = '/home/hocheol/inskin_ai/no_track/datasets/age_data/v0.3/v0.3_data/114_remained'

file_paths = []
filtered_paths = []

# 안면인식데이터(114_data)에서 나이 라벨이 있는 user_id 중 114_filtered에 존재하지 않는 file_paths에 저장
for user_id in tqdm(merged_ids):
    user_dir = os.path.join(ORIGIN_ROOT, user_id)
    for root, dirs, files in os.walk(user_dir):
        for file in files:
            if file.endswith('.json'):
                continue
            filtered_path = os.path.join(FILTERED_ROOT, file)
            origin_file_path = os.path.join(root, file)

            remained_file_path = os.path.join(REMAINED_ROOT, user_id, file)
            if not os.path.exists(filtered_path):
                file_paths.append((origin_file_path, remained_file_path))
            else:
                filtered_paths.append(filtered_path)

len(file_paths), len(filtered_paths)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1247/1247 [00:03<00:00, 397.94it/s]


(164898, 59562)

In [10]:
paths = []
for pair in file_paths:
    _, path = pair
    if not os.path.exists(path):
        print(f"Missing: {path}")
    else:
        paths.append(path)
len(paths)

Missing: /home/hocheol/inskin_ai/no_track/datasets/age_data/v0.3/v0.3_data/114_remained/1032/1032_Tablet_Low_008.jpg
Missing: /home/hocheol/inskin_ai/no_track/datasets/age_data/v0.3/v0.3_data/114_remained/2438/2438_Tablet_Low_023.jpg
Missing: /home/hocheol/inskin_ai/no_track/datasets/age_data/v0.3/v0.3_data/114_remained/2438/2438_Tablet_Low_019.jpg
Missing: /home/hocheol/inskin_ai/no_track/datasets/age_data/v0.3/v0.3_data/114_remained/2438/2438_Tablet_Low_022.jpg
Missing: /home/hocheol/inskin_ai/no_track/datasets/age_data/v0.3/v0.3_data/114_remained/2438/2438_Tablet_Low_021.jpg
Missing: /home/hocheol/inskin_ai/no_track/datasets/age_data/v0.3/v0.3_data/114_remained/2101/2101_Tablet_Mid_027.jpg
Missing: /home/hocheol/inskin_ai/no_track/datasets/age_data/v0.3/v0.3_data/114_remained/2146/2146_Tablet_Low_028.jpg
Missing: /home/hocheol/inskin_ai/no_track/datasets/age_data/v0.3/v0.3_data/114_remained/2146/2146_Phone_Low_027.jpg
Missing: /home/hocheol/inskin_ai/no_track/datasets/age_data/v0.3/

164187

In [19]:
user_info = {}
for phase in ['train', 'val', 'test']:
    for item in tqdm(data_dict[phase]):
        user_id = item['user_id']
        path = ''
        gender = item['gender']
        age = item['age']
        age_class = item['age_class']
        size = {'height':-1, 'width':-1, 'channels':-1}
        if user_id not in user_info.keys():
            user_info[user_id] = {
                'user_id': user_id,
                'path': path,
                'gender': gender,
                'age': age,
                'age_class': age_class,
                'size': size
            }
len(user_info)

  0%|                                                                                                                              | 0/47527 [00:00<?, ?it/s]

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6166/6166 [00:00<00:00, 680831.84it/s]


1247

In [28]:
new_data_dict = {'train': [], 'val': [], 'test': []}

for path in tqdm(paths):
    user_id = path.split('/')[-2]
    
    user_data = user_info[user_id]
    user_data['path'] = path
    h, w, c = cv2.imread(path).shape
    user_data['size'] = {'height': h, 'width': w, 'channels': c}

    if user_id in id_sets['train']:
        # print('train')
        new_data_dict['train'].append(user_data)
    elif user_id in id_sets['val']:
        # print('val')
        new_data_dict['val'].append(user_data)
    elif user_id in id_sets['test']:
        # print('test')
        new_data_dict['test'].append(user_data)
    else:
        print(f"User ID {user_id} not found in any set.")

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 164187/164187 [10:18<00:00, 265.26it/s]


In [31]:
new_label_path = '/home/hocheol/inskin_ai/no_track/datasets/age_data/v0.3/v0.3.0/114_remained/label'
os.makedirs(new_label_path)
for phase in ['train', 'val', 'test']:
    with open(os.path.join(new_label_path, f"age_{phase}.json"), 'w') as f:
        json.dump(new_data_dict[phase], f, indent=2, ensure_ascii=False)